# Analisis de resultados con tc

Este notebook analiza los logs de `resultados_tc`, donde se probaron distintos escenarios de red:

- Sin limitacion.
- Limitacion de ancho de banda.
- Retardo artificial.
- Perdida artificial.
- Red degradada con delay + loss.

El objetivo es generar tablas y graficas para explicar como cambia el RTT y si los mensajes de G1/G2 siguen recibiendo ACK bajo cada condicion.

In [ ]:
# Ejecuta esta celda solo si faltan dependencias.
# %pip install pandas matplotlib

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd

BASE_DIR = Path.cwd()

# Permite ejecutar desde esta carpeta o desde la raiz del repo.
if any(BASE_DIR.glob("*.log")):
    TC_DIR = BASE_DIR
elif (BASE_DIR / "host" / "métricas_antiguas" / "resultados_tc").exists():
    TC_DIR = BASE_DIR / "host" / "métricas_antiguas" / "resultados_tc"
else:
    raise FileNotFoundError("No se encuentra la carpeta resultados_tc")

OUT_DIR = TC_DIR / "analisis"
PLOTS_DIR = OUT_DIR / "graficas"
OUT_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

TC_DIR

## 1. Parseo de logs

Se extraen de cada log:

- Escenario probado.
- Regla `tc` aplicada.
- RTT minimo/medio/maximo a M2 y M3.
- Perdida observada por ping.
- ACK recibido al enviar mensaje a G1 y G2.
- Si el mensaje aparece en el historico del grupo.

In [ ]:
SCENARIO_LABELS = {
    "01_sin_limitacion": "Sin limitacion",
    "02_ancho_banda_10mbit": "BW 10 Mbit",
    "03_ancho_banda_1mbit": "BW 1 Mbit",
    "04_delay_100ms": "Delay 100 ms",
    "05_delay_200ms": "Delay 200 ms",
    "06_loss_2porciento": "Loss 2%",
    "07_loss_5porciento": "Loss 5%",
    "08_red_mala_delay_loss": "Delay 100 ms + loss 2%",
}

def parse_log(path: Path):
    text = path.read_text(encoding="utf-8", errors="ignore")
    scenario = path.stem
    order = int(scenario.split("_", 1)[0])
    label = SCENARIO_LABELS.get(scenario, scenario)

    date_match = re.search(r"FECHA:\s*(.+)", text)
    rule_match = re.search(r"Aplicando regla tc:\n(.+)", text)
    qdisc_match = re.search(r"\[3\] Estado de tc:\n(.+)", text)

    rule = rule_match.group(1).strip() if rule_match else "sin_limitacion"
    qdisc = qdisc_match.group(1).strip() if qdisc_match else ""

    ping_rows = []
    ping_blocks = re.findall(
        r"\[(?:4|5)\] Ping IPv6 a (M[23]):.*?"
        r"(\d+) packets transmitted, (\d+) received, ([\d.]+)% packet loss.*?"
        r"rtt min/avg/max/mdev = ([\d.]+)/([\d.]+)/([\d.]+)/([\d.]+) ms",
        text,
        flags=re.S,
    )

    for dest, tx, rx, loss, rtt_min, rtt_avg, rtt_max, mdev in ping_blocks:
        ping_rows.append({
            "orden": order,
            "escenario": scenario,
            "escenario_label": label,
            "fecha": date_match.group(1).strip() if date_match else "",
            "regla_tc": rule,
            "qdisc": qdisc,
            "destino": dest,
            "paquetes_tx": int(tx),
            "paquetes_rx": int(rx),
            "perdida_ping_pct": float(loss),
            "rtt_min_ms": float(rtt_min),
            "rtt_media_ms": float(rtt_avg),
            "rtt_max_ms": float(rtt_max),
            "mdev_ms": float(mdev),
        })

    ack_rows = []
    for group in [1, 2]:
        client_id = f"test-{scenario}-g{group}"
        ack_ok = client_id in text and "ACK recibido: OK mensaje_recibido" in text
        in_history = re.search(rf"Histórico grupo {group}:.*?client_id={re.escape(client_id)}", text, flags=re.S) is not None
        ack_rows.append({
            "orden": order,
            "escenario": scenario,
            "escenario_label": label,
            "grupo": f"G{group}",
            "client_id": client_id,
            "ack_ok": ack_ok,
            "aparece_historico": in_history,
        })

    return ping_rows, ack_rows

ping_rows = []
ack_rows = []
for path in sorted(TC_DIR.glob("*.log")):
    p_rows, a_rows = parse_log(path)
    ping_rows.extend(p_rows)
    ack_rows.extend(a_rows)

ping_df = pd.DataFrame(ping_rows).sort_values(["orden", "destino"])
ack_df = pd.DataFrame(ack_rows).sort_values(["orden", "grupo"])

ping_df.to_csv(OUT_DIR / "resumen_ping_tc.csv", index=False)
ack_df.to_csv(OUT_DIR / "resumen_ack_historico_tc.csv", index=False)

ping_df

In [ ]:
ack_df

## 2. RTT medio por escenario

Esta grafica muestra como cambia el RTT medio hacia M2 y M3 segun la regla `tc` aplicada.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
labels = ping_df.sort_values("orden")["escenario_label"].drop_duplicates().tolist()
x_base = list(range(len(labels)))
styles = {
    "M2": {"offset": -0.06, "marker": "o", "linestyle": "--", "color": "#1971c2"},
    "M3": {"offset": 0.06, "marker": "s", "linestyle": "-", "color": "#f08c00"},
}

for dest, dest_df in ping_df.groupby("destino"):
    dest_df = dest_df.sort_values("orden")
    style = styles.get(dest, {})
    x = [i + style.get("offset", 0) for i in x_base]
    ax.plot(
        x,
        dest_df["rtt_media_ms"],
        marker=style.get("marker", "o"),
        linestyle=style.get("linestyle", "-"),
        color=style.get("color"),
        linewidth=2,
        label=dest,
    )

ax.set_title("RTT medio bajo escenarios tc")
ax.set_xlabel("Escenario")
ax.set_ylabel("RTT medio (ms)")
ax.set_xticks(x_base)
ax.set_xticklabels(labels, rotation=35, ha="right")
ax.legend(title="Destino")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "01_rtt_medio_por_escenario.png", dpi=160)
plt.show()

**Interpretacion esperada:** los escenarios `Delay 100 ms`, `Delay 200 ms` y `Delay 100 ms + loss 2%` deben elevar el RTT aproximadamente a esos valores. Las limitaciones de ancho de banda no tienen por que aumentar mucho el ping, porque ICMP usa paquetes pequenos.

## 3. RTT minimo, medio y maximo

Esta grafica permite ver no solo el promedio, sino tambien la variabilidad entre minimo y maximo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, dest in zip(axes, ["M2", "M3"]):
    df = ping_df[ping_df["destino"] == dest].sort_values("orden")
    x = range(len(df))
    ax.bar(x, df["rtt_media_ms"], color="#1971c2", label="Media")
    ax.errorbar(
        x,
        df["rtt_media_ms"],
        yerr=[df["rtt_media_ms"] - df["rtt_min_ms"], df["rtt_max_ms"] - df["rtt_media_ms"]],
        fmt="none",
        color="black",
        capsize=4,
        label="Min-max",
    )
    ax.set_title(dest)
    ax.set_xticks(list(x))
    ax.set_xticklabels(df["escenario_label"], rotation=45, ha="right")
    ax.set_ylabel("RTT (ms)")
    ax.legend()

fig.suptitle("RTT minimo, medio y maximo por destino")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "02_rtt_min_media_max.png", dpi=160)
plt.show()

## 4. Perdida observada con ping

Esta grafica muestra la perdida observada por `ping` en cada escenario.

In [ ]:
loss_pivot = ping_df.pivot(index="escenario_label", columns="destino", values="perdida_ping_pct")
loss_pivot = loss_pivot.loc[ping_df.sort_values("orden")["escenario_label"].drop_duplicates()]

fig, ax = plt.subplots(figsize=(13, 5))
loss_pivot.plot(kind="bar", ax=ax, color=["#1971c2", "#f08c00"])
ax.set_title("Perdida observada con ping")
ax.set_xlabel("Escenario")
ax.set_ylabel("Perdida (%)")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "03_perdida_ping_por_escenario.png", dpi=160)
plt.show()

**Nota importante:** si la perdida sale 0% incluso en escenarios `loss 2%` o `loss 5%`, no significa que la regla no funcionase. Cada prueba usa solo 10 paquetes ICMP, por lo que la muestra es pequena. Para medir perdida con mas fiabilidad conviene aumentar el numero de pings o usar UDP/iperf.

## 5. ACK e historico de mensajes por grupo

Esta parte comprueba si, bajo cada condicion de red, el envio a G1 y G2 recibio ACK y aparecio en el historico del grupo.

In [ ]:
ack_plot = ack_df.copy()
ack_plot["ack_val"] = ack_plot["ack_ok"].astype(int)
ack_plot["hist_val"] = ack_plot["aparece_historico"].astype(int)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, group in zip(axes, ["G1", "G2"]):
    df = ack_plot[ack_plot["grupo"] == group].sort_values("orden")
    x = range(len(df))
    ax.bar([i - 0.18 for i in x], df["ack_val"], width=0.36, label="ACK", color="#2f9e44")
    ax.bar([i + 0.18 for i in x], df["hist_val"], width=0.36, label="Historico", color="#1971c2")
    ax.set_title(group)
    ax.set_xticks(list(x))
    ax.set_xticklabels(df["escenario_label"], rotation=45, ha="right")
    ax.set_ylim(0, 1.15)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["No", "Si"])
    ax.legend()

fig.suptitle("ACK e historico de mensajes bajo escenarios tc")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "04_ack_historico_por_grupo.png", dpi=160)
plt.show()

**Interpretacion:** si las barras estan en `Si`, el cliente recibio ACK y el mensaje aparece en el historico. Esto permite defender que, en estas pruebas unitarias por escenario, el servicio siguio respondiendo bajo las condiciones de red aplicadas.

## 6. Tabla final para la presentacion

Resumen compacto por escenario, agregando M2/M3.

In [ ]:
scenario_summary = (
    ping_df.groupby(["orden", "escenario", "escenario_label", "regla_tc"], as_index=False)
    .agg(
        rtt_medio_global_ms=("rtt_media_ms", "mean"),
        rtt_max_global_ms=("rtt_max_ms", "max"),
        perdida_media_ping_pct=("perdida_ping_pct", "mean"),
    )
    .sort_values("orden")
)

ack_summary = (
    ack_df.groupby(["orden", "escenario"], as_index=False)
    .agg(
        grupos_con_ack=("ack_ok", "sum"),
        grupos_en_historico=("aparece_historico", "sum"),
    )
)

final_summary = scenario_summary.merge(ack_summary, on=["orden", "escenario"], how="left")
final_summary["rtt_medio_global_ms"] = final_summary["rtt_medio_global_ms"].round(3)
final_summary["rtt_max_global_ms"] = final_summary["rtt_max_global_ms"].round(3)
final_summary["perdida_media_ping_pct"] = final_summary["perdida_media_ping_pct"].round(2)
final_summary.to_csv(OUT_DIR / "resumen_final_tc.csv", index=False)
final_summary

## Conclusiones

- El escenario sin limitacion presenta RTT muy bajo.
- Las limitaciones de ancho de banda apenas afectan al RTT de ping porque los paquetes ICMP son pequenos.
- Los escenarios con delay muestran claramente el incremento esperado de latencia.
- La perdida observada con ping puede salir 0% por usar pocas muestras; para defender perdida conviene complementarlo con iperf UDP o repetir ping con mas paquetes.
- Los envios a G1 y G2 reciben ACK y aparecen en historico en los escenarios analizados, por lo que el servicio sigue respondiendo en estas pruebas puntuales.